In [ ]:
import pandas as pd
import requests
from io import BytesIO

# GitHub raw base URL for the folder
base_url = "https://raw.githubusercontent.com/utexas-SDS-357/sds357-project-sp26-the-grazing-goats/main/data/NC_FBI_UCR_RAW_DATA/"

file_names = [
    "US-2000.xls", "US-2001.xls", "US-2002.xls", "US-2003.xls",
    "US-2004.xls", "US-2005.xls", "north-carolina-2006.xls",
    "north-carolina-2007.xls", "north-carolina-2008.xls", "north-carolina-2009.xls",
    "north-carolina-2010.xls", "north-carolina-2011.xls", "north-carolina-2012.xls",
    "north-carolina-2013.xls", "north-carolina-2014.xls", "north-carolina-2015.xls"
]

run_throughs = 0
for file_name in file_names:
    if 'north' in file_name.lower():
        file_url = base_url + file_name
        response = requests.get(file_url)
        file_bytes = BytesIO(response.content)

        if run_throughs == 0:
            df = pd.read_excel(file_bytes)
            header_row = df[df.iloc[:,0].astype(str).str.strip() == 'City'].index[0]
            df = df.iloc[header_row:].reset_index(drop=True)
            df.columns = df.iloc[0]
            df = df[1:].reset_index(drop=True)
            df['Year'] = file_name.split('-')[-1].split('.')[0]
            run_throughs = 1
        else:
            df2 = pd.read_excel(file_bytes)
            try:
                header_row = df2[df2.iloc[:,0].astype(str).str.strip() == 'City'].index[0]
                df2 = df2.iloc[header_row:].reset_index(drop=True)
                df2.columns = df2.iloc[0]
                df2 = df2[1:].reset_index(drop=True)
                df2['Year'] = file_name.split('-')[-1].split('.')[0]
                df = pd.concat([df, df2])
            except:
                print(file_name)

fix_map = {'Violent crime': ['Violent\ncrime'],
    'Murder and nonnegligent manslaughter': ['Murder and\nnonnegligent\nmanslaughter'],
    'Forcible rape': ['Forcible\nrape'],
    'Aggravated assault': ['Aggravated\nassault'],
    'Property crime': ['Property\ncrime'],
    'Larceny-theft': ['Larceny-\ntheft'],
    'Motor vehicle theft': ['Motor\nvehicle\ntheft'],
    'Arson': ['Arson1'],
    'Rape': ['Rape2', 'Rape\n(revised\ndefinition)1', 'Rape\n(legacy\ndefinition)2']}

for correct, broken_list in fix_map.items():
    for broken in broken_list:
        if correct in df.columns and broken in df.columns:
            df[correct] = df[correct].fillna(df[broken])

cols_to_drop = [c for v in fix_map.values() for c in v if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

df[df['City'].isin(["Charlotte", 'Charlotte-Mecklenburg', "Durham", 'Durham2', "Fayetteville", "Greensboro", "Raleigh", "Winston-Salem"])].sort_values('City')[['City', 'Year']].value_counts()
cities_df = df[df['City'].isin(["Charlotte", 'Charlotte-Mecklenburg', "Durham", 'Durham2', "Fayetteville", "Greensboro", "Raleigh", "Winston-Salem"])]
cities_df['City'] = cities_df['City'].replace({'Charlotte-Mecklenburg': 'Charlotte', 'Durham2': 'Durham'})
cities_df.sort_values('City')[['City', 'Year']].value_counts()
cities_df['City'].value_counts()

cities_df.to_csv('NC_FBI_crime_data_clean.csv', index=False)